# Influence of Unbalance Phase on Cracked Rotor Vibration

In a cracked rotor, the **breathing mechanism** — the periodic opening and
closing of the crack as the shaft rotates — depends on the stress field at the
crack front. When residual unbalance is present, its centrifugal force adds to
(or subtracts from) the gravity-driven bending stress that governs crack
opening. The **angular position of the unbalance mass relative to the crack**
therefore modulates:

1. **When** the crack opens during each revolution.
2. **How deeply** it opens (effective breathing amplitude).
3. The resulting **harmonic content** of the vibration response.

If the unbalance mass is aligned so that it forces the crack *open* at the same
instant gravity does, the 2X and higher harmonics are amplified. Conversely,
when the unbalance opposes the crack opening, the breathing effect is partially
suppressed and the spectrum approaches that of a healthy rotor.

This notebook sweeps the unbalance phase from 0 to 2π in 45° steps and
compares the **orbit shapes** and **frequency spectra** at each phase to
quantify this interaction.

In [26]:
import numpy as np
import ross as rs
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"
print(f"ROSS version: {rs.__version__}")

from constants import *

Q_ = rs.Q_

rotor = rs.Rotor.load("sinha_rotor.toml")

ROSS version: 2.2.0


## Simulation Configuration

Sweep the unbalance phase from 0 to 2π in 45° increments while keeping the
unbalance magnitude, crack depth, rotor speed, and simulation time fixed.

In [27]:
probe = [rs.Probe(node=PROBE_NODE, angle=0.0)]

PHASES_DEG = np.arange(0, 360, 15)
PHASES_RAD = np.deg2rad(PHASES_DEG)

print(f"Disk node: {DISK_NODE}")
print(f"Crack node: {CRACK_NODE}")
print(f"Probe node: {PROBE_NODE}")
print(f"Phases (deg): {PHASES_DEG}")
for spd in SPEEDS:
    print(f"Speed: {spd}")

Disk node: 6
Crack node: 7
Probe node: 10
Phases (deg): [  0  15  30  45  60  75  90 105 120 135 150 165 180 195 210 225 240 255
 270 285 300 315 330 345]
Speed: 650 revolutions_per_minute
Speed: 750 revolutions_per_minute


In [28]:
results = {}
unb_mag_val = UNB_MAG.to("kg*m").m

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_val = speed.to("rad/s").m
    results[speed_rpm] = {}

    for phase_deg, phase_rad in zip(PHASES_DEG, PHASES_RAD):
        print(f"Running crack simulation: {speed_rpm} RPM, phase = {phase_deg}°...")
        res = rotor.run_crack(
            n=CRACK_NODE,
            depth_ratio=CRACK_RATIO,
            crack_model="Gasch",
            node=[DISK_NODE],
            unbalance_magnitude=[unb_mag_val],
            unbalance_phase=[phase_rad],
            speed=speed_val,
            t=T,
            model_reduction={"num_modes": 12},
        )
        results[speed_rpm][phase_deg] = res

total = sum(len(v) for v in results.values())
print(f"\nCompleted {total} simulations across {len(SPEEDS)} speeds.")

Running crack simulation: 650 RPM, phase = 0°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 15°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 30°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 45°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 60°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 75°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 90°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 105°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 120°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 135°...
Running with model reduction: pseudomodal
Running crack simulation: 650 RPM, phase = 150°...
Running with model reductio

## Orbit Plots

The orbit at the probe node reveals how the unbalance phase reshapes the
vibration trajectory. A purely circular orbit indicates dominant 1X content; a
figure-eight or multi-lobed shape signals strong 2X and higher harmonics
introduced by the breathing crack.

We use the last portion of the time record (steady-state) to plot orbits for
each phase on a common subplot grid.

In [29]:
ndof = rotor.number_dof
nodes = rotor.nodes
link_nodes = rotor.link_nodes

fix_dof = (PROBE_NODE - nodes[-1] - 1) * ndof // 2 if PROBE_NODE in link_nodes else 0
dofx = ndof * PROBE_NODE - fix_dof
dofy = ndof * PROBE_NODE + 1 - fix_dof

steady_start = int(len(T) * 0.5)

n_phases = len(PHASES_DEG)
ncols = 4
nrows = int(np.ceil(n_phases / ncols))

colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
]

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)

    fig_orbits = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"φ = {p}°" for p in PHASES_DEG],
        horizontal_spacing=0.08,
        vertical_spacing=0.05,
    )

    for i, phase_deg in enumerate(PHASES_DEG):
        row = i // ncols + 1
        col = i % ncols + 1
        res = results[speed_rpm][phase_deg]
        x_ss = res.yout[steady_start:, dofx] * 1e6
        y_ss = res.yout[steady_start:, dofy] * 1e6

        fig_orbits.add_trace(
            go.Scatter(
                x=x_ss, y=y_ss,
                mode="lines",
                line=dict(color=colors[i % len(colors)], width=1),
                name=f"φ = {phase_deg}°",
                showlegend=False,
            ),
            row=row, col=col,
        )
        fig_orbits.update_xaxes(title_text="X (μm)", row=row, col=col)
        fig_orbits.update_yaxes(title_text="Y (μm)", row=row, col=col, scaleanchor=f"x{i+1}" if i > 0 else "x")

    fig_orbits.update_layout(
        title=dict(text=f"Orbit at Probe Node {PROBE_NODE} — Crack Depth {CRACK_RATIO*100:.0f}%, Speed {speed_rpm} RPM"),
        height=300 * nrows,
        width=1100,
    )
    fig_orbits.show()

Phase combinations most similar to Figure 3 in Sinha et al. (2007).
 
Phases: 240° and 315°

## Frequency Spectra (DFFT)

The discrete FFT of the steady-state response highlights how the unbalance
phase redistributes energy among harmonics. The 1X peak (synchronous) is always
present due to the unbalance itself. The 2X and 3X peaks are the crack
signature — their amplitudes vary with the phase angle.

In [38]:
# TODO
# Check this DFFT implementation. Is this the power FFT?
# Is this the correct way to do it?
# Is it the same in ross?


freq_range_hz = FREQ_RANGE.to("Hz").m

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m

    fig_spectra = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"φ = {p}°" for p in PHASES_DEG],
        horizontal_spacing=0.08,
        vertical_spacing=0.05,
    )

    for i, phase_deg in enumerate(PHASES_DEG):
        row = i // ncols + 1
        col = i % ncols + 1
        res = results[speed_rpm][phase_deg]

        x_ss = res.yout[steady_start:, dofx]
        dt = T[1] - T[0]
        N = len(x_ss)
        freqs = np.fft.rfftfreq(N, d=dt)
        fft_amp = 2.0 / N * np.abs(np.fft.rfft(x_ss))

        mask = (freqs >= freq_range_hz[0]) & (freqs <= freq_range_hz[1])

        fig_spectra.add_trace(
            go.Scatter(
                x=freqs[mask],
                y=fft_amp[mask] * 1e6,
                mode="lines",
                line=dict(color=colors[i % len(colors)], width=1),
                name=f"φ = {phase_deg}°",
                showlegend=False,
            ),
            row=row, col=col,
        )

        for nx, ls in [(1, "solid"), (2, "dash"), (3, "dot")]:
            fig_spectra.add_vline(
                x=nx * speed_hz,
                line=dict(color="gray", width=0.8, dash=ls),
                annotation_text=f"{nx}X",
                annotation_position="top",
                row=row, col=col,
            )

        fig_spectra.update_xaxes(title_text="Frequency (Hz)", row=row, col=col)
        fig_spectra.update_yaxes(title_text="Amplitude (μm)", row=row, col=col)

    fig_spectra.update_layout(
        title=dict(text=f"Frequency Spectrum at Probe Node {PROBE_NODE} — Crack Depth {CRACK_RATIO*100:.0f}%, Speed {speed_rpm} RPM"),
        height=300 * nrows,
        width=1100,
    )
    fig_spectra.show()

## Overlaid Spectra

Plotting all phase cases on a single frequency-domain figure makes the
harmonic amplitude variation immediately visible.

In [31]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m

    fig_overlay = go.Figure()

    for i, phase_deg in enumerate(PHASES_DEG):
        res = results[speed_rpm][phase_deg]
        x_ss = res.yout[steady_start:, dofx]
        dt = T[1] - T[0]
        N = len(x_ss)
        freqs = np.fft.rfftfreq(N, d=dt)
        fft_amp = 2.0 / N * np.abs(np.fft.rfft(x_ss))
        mask = (freqs >= freq_range_hz[0]) & (freqs <= freq_range_hz[1])

        fig_overlay.add_trace(
            go.Scatter(
                x=freqs[mask],
                y=fft_amp[mask] * 1e6,
                mode="lines",
                name=f"φ = {phase_deg}°",
                line=dict(width=1.5),
            )
        )

    for nx in [1, 2, 3]:
        fig_overlay.add_vline(
            x=nx * speed_hz,
            line=dict(color="gray", width=1, dash="dash"),
            annotation_text=f"{nx}X",
            annotation_position="top right",
        )

    fig_overlay.update_layout(
        title=dict(text=f"Overlaid Frequency Spectra — {speed_rpm} RPM"),
        xaxis_title="Frequency (Hz)",
        yaxis_title="Amplitude (μm)",
        height=500,
        width=1000,
        legend=dict(title="Unbalance Phase"),
    )
    fig_overlay.show()

## Harmonic Amplitude vs Unbalance Phase

Extract the 1X, 2X and 3X amplitudes from each simulation and plot them as a
function of unbalance phase. This polar/bar representation summarises how the
crack breathing couples with the unbalance orientation.

In [32]:
import pandas as pd


def get_harmonic_amplitude(yout, dof, t, target_freq_hz):
    """Extract the FFT amplitude at the frequency bin closest to target_freq_hz."""
    signal = yout[steady_start:, dof]
    dt = t[1] - t[0]
    N = len(signal)
    freqs = np.fft.rfftfreq(N, d=dt)
    fft_vals = 2.0 / N * np.abs(np.fft.rfft(signal))
    idx = np.argmin(np.abs(freqs - target_freq_hz))
    return fft_vals[idx]


dfs = {}

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m
    harmonics_data = {"phase_deg": [], "1X": [], "2X": [], "3X": []}

    for phase_deg in PHASES_DEG:
        res = results[speed_rpm][phase_deg]
        harmonics_data["phase_deg"].append(phase_deg)
        harmonics_data["1X"].append(get_harmonic_amplitude(res.yout, dofx, T, speed_hz) * 1e6)
        harmonics_data["2X"].append(get_harmonic_amplitude(res.yout, dofx, T, 2 * speed_hz) * 1e6)
        harmonics_data["3X"].append(get_harmonic_amplitude(res.yout, dofx, T, 3 * speed_hz) * 1e6)

    dfs[speed_rpm] = pd.DataFrame(harmonics_data)
    print(f"\n--- {speed_rpm} RPM ---")
    print(dfs[speed_rpm].to_string(index=False, float_format="{:.4f}".format))


--- 650 RPM ---
 phase_deg      1X     2X     3X
         0 13.1328 0.4501 0.6002
        15 13.1462 0.1883 0.5698
        30 13.1812 0.5204 0.5242
        45 13.2276 0.9179 0.4733
        60 13.2721 1.2626 0.4309
        75 13.3024 1.5238 0.4123
        90 13.3102 1.6820 0.4257
       105 13.2934 1.7258 0.4649
       120 13.2565 1.6521 0.5154
       135 13.2098 1.4662 0.5624
       150 13.1662 1.1814 0.5955
       165 13.1383 0.8194 0.6088
       180 13.1345 0.4169 0.6000
       195 13.1567 0.1977 0.5708
       210 13.1998 0.5537 0.5265
       225 13.2528 0.9529 0.4770
       240 13.3023 1.2981 0.4360
       255 13.3355 1.5595 0.4183
       270 13.3441 1.7177 0.4318
       285 13.3257 1.7616 0.4704
       300 13.2852 1.6879 0.5198
       315 13.2328 1.5020 0.5658
       330 13.1819 1.2171 0.5979
       345 13.1456 0.8548 0.6101

--- 750 RPM ---
 phase_deg      1X     2X     3X
         0 12.1357 1.6358 0.0924
        15 12.1570 1.1464 0.0698
        30 12.3185 3.1964 0.0800
        4

In [39]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    df = dfs[speed_rpm]

    fig_bar = go.Figure()

    bar_width = 12
    offsets = [-bar_width, 0, bar_width]

    for harmonic, offset, color in zip(
        ["1X", "2X", "3X"],
        offsets,
        ["#1f77b4", "#ff7f0e", "#2ca02c"],
    ):
        fig_bar.add_trace(
            go.Bar(
                x=[p + offset for p in df["phase_deg"]],
                y=df[harmonic],
                name=harmonic,
                marker_color=color,
                width=bar_width,
            )
        )

    fig_bar.update_layout(
        title=dict(text=f"Harmonic Amplitudes vs Unbalance Phase — {speed_rpm} RPM"),
        xaxis_title="Unbalance Phase (°)",
        yaxis_title="Amplitude (μm)",
        barmode="group",
        xaxis=dict(tickvals=list(PHASES_DEG), ticktext=[f"{p}°" for p in PHASES_DEG]),
        height=450,
        width=900,
        legend=dict(title="Harmonic"),
    )
    fig_bar.show()

In [40]:
theta_closed = np.append(PHASES_DEG, PHASES_DEG[0])

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    df = dfs[speed_rpm]

    fig_polar = go.Figure()

    for harmonic, color in zip(["1X", "2X", "3X"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
        r_vals = np.append(df[harmonic].values, df[harmonic].values[0])
        fig_polar.add_trace(
            go.Scatterpolar(
                r=r_vals,
                theta=theta_closed,
                mode="lines+markers",
                name=harmonic,
                line=dict(color=color, width=2),
                marker=dict(size=6),
            )
        )

    fig_polar.update_layout(
        title=dict(text=f"Harmonic Amplitudes — Polar Plot — {speed_rpm} RPM"),
        polar=dict(
            angularaxis=dict(
                tickvals=list(PHASES_DEG),
                ticktext=[f"{p}°" for p in PHASES_DEG],
                direction="counterclockwise",
            ),
            radialaxis=dict(title="Amplitude (μm)"),
        ),
        height=550,
        width=650,
        legend=dict(title="Harmonic"),
    )
    fig_polar.show()

## Steady-State Time Waveforms

Compare the time-domain vibration waveform over two shaft revolutions for
selected unbalance phases. Waveform distortion (departure from a pure sine)
increases when the crack-induced harmonics are amplified.

In [41]:
n_revs = 2

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m
    period = 1.0 / speed_hz
    t_window = n_revs * period
    t_start_ss = T[steady_start]
    t_end_window = t_start_ss + t_window
    mask_time = (T >= t_start_ss) & (T <= t_end_window)

    fig_time = go.Figure()

    for i, phase_deg in enumerate(PHASES_DEG):
        res = results[speed_rpm][phase_deg]
        fig_time.add_trace(
            go.Scatter(
                x=(T[mask_time] - t_start_ss) / period,
                y=res.yout[mask_time, dofx] * 1e6,
                mode="lines",
                name=f"φ = {phase_deg}°",
                line=dict(width=1.5),
            )
        )

    fig_time.update_layout(
        title=dict(text=f"Steady-State Waveform ({n_revs} Revolutions) — X Direction — {speed_rpm} RPM"),
        xaxis_title="Revolutions",
        yaxis_title="Amplitude (μm)",
        height=450,
        width=1000,
        legend=dict(title="Unbalance Phase"),
    )
    fig_time.show()

## Orbit Comparison at the Disk Node

The disk node experiences the largest deflections. Plotting orbits there shows
the structural influence of the crack more prominently than at the probe
location.

In [43]:
fix_dof_disk = (DISK_NODE - nodes[-1] - 1) * ndof // 2 if DISK_NODE in link_nodes else 0
dofx_disk = ndof * DISK_NODE - fix_dof_disk
dofy_disk = ndof * DISK_NODE + 1 - fix_dof_disk

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)

    fig_orbits_disk = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"φ = {p}°" for p in PHASES_DEG],
        horizontal_spacing=0.08,
        vertical_spacing=0.05,
    )

    for i, phase_deg in enumerate(PHASES_DEG):
        row = i // ncols + 1
        col = i % ncols + 1
        res = results[speed_rpm][phase_deg]
        x_ss = res.yout[steady_start:, dofx_disk] * 1e6
        y_ss = res.yout[steady_start:, dofy_disk] * 1e6

        fig_orbits_disk.add_trace(
            go.Scatter(
                x=x_ss, y=y_ss,
                mode="lines",
                line=dict(color=colors[i % len(colors)], width=1),
                name=f"φ = {phase_deg}°",
                showlegend=False,
            ),
            row=row, col=col,
        )
        fig_orbits_disk.update_xaxes(title_text="X (μm)", row=row, col=col)
        fig_orbits_disk.update_yaxes(title_text="Y (μm)", row=row, col=col)

    fig_orbits_disk.update_layout(
        title=dict(text=f"Orbit at Disk Node {DISK_NODE} — Crack Depth {CRACK_RATIO*100:.0f}%, Speed {speed_rpm} RPM"),
        height=300 * nrows,
        width=1100,
    )
    fig_orbits_disk.show()

## 2X / 1X Amplitude Ratio

The ratio of the 2X harmonic to the 1X harmonic is a key diagnostic indicator
for breathing cracks. Plotting this ratio against the unbalance phase shows how
sensitive crack detection is to the relative orientation of the unbalance mass.

In [44]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    df = dfs[speed_rpm]
    df["2X/1X"] = df["2X"] / df["1X"]
    df["3X/1X"] = df["3X"] / df["1X"]

    fig_ratio = go.Figure()

    fig_ratio.add_trace(
        go.Scatter(
            x=df["phase_deg"],
            y=df["2X/1X"],
            mode="lines+markers",
            name="2X / 1X",
            line=dict(color="#ff7f0e", width=2),
            marker=dict(size=8),
        )
    )
    fig_ratio.add_trace(
        go.Scatter(
            x=df["phase_deg"],
            y=df["3X/1X"],
            mode="lines+markers",
            name="3X / 1X",
            line=dict(color="#2ca02c", width=2),
            marker=dict(size=8),
        )
    )

    fig_ratio.update_layout(
        title=dict(text=f"Harmonic Ratios vs Unbalance Phase — {speed_rpm} RPM"),
        xaxis_title="Unbalance Phase (°)",
        yaxis_title="Amplitude Ratio",
        xaxis=dict(tickvals=list(PHASES_DEG), ticktext=[f"{p}°" for p in PHASES_DEG]),
        height=400,
        width=800,
        legend=dict(title="Ratio"),
    )
    fig_ratio.show()

## Observations

1. **Orbit shape** changes significantly with unbalance phase. Certain phases
   produce multi-lobed or figure-eight orbits (strong 2X content), while others
   yield nearly circular trajectories (suppressed crack breathing).

2. **2X harmonic amplitude** is maximised when the unbalance force reinforces
   the gravity-driven crack opening and minimised when it opposes it.

3. The **1X component** also varies because the effective dynamic stiffness of
   the cracked cross-section changes with the breathing pattern, which is
   itself modulated by the unbalance phase.

4. **Diagnostic implication**: crack detection methods based solely on 2X
   amplitude may miss a crack if the unbalance phase happens to suppress the
   breathing effect. Monitoring the *ratio* 2X/1X across multiple operating
   conditions (e.g. different balancing states) improves detection robustness.

5. The **polar plot** of 2X amplitude versus phase angle reveals the
   directional sensitivity of the crack signature — useful for estimating the
   angular position of the crack relative to the unbalance.

## Crack Simulations

Run a breathing-crack simulation (Mayes model) for each unbalance phase.
The crack is located one element downstream of the disk (node 7, at 315 mm).